# 1. Data Cleaning and Preparation

In [7]:
import pandas as pd

# Load the raw dataset
df = pd.read_csv("../data/raw/statsbomb_non_penalty_shots_raw.csv")

# Preserve a copy of the original dataset
df_raw = df.copy()

print("Dataset shape:", df.shape)
display(df.head())
df.info()


Dataset shape: (12374, 13)


,match_id,id,team,player,location,shot_body_part,shot_technique,shot_first_time,shot_type,play_pattern,under_pressure,shot_outcome,shot_statsbomb_xg
0,3939988,20be3941-09d4-4073-98f7-a56806520e3e,Mexico,Orbelín Pineda Alvarado,"[96.5, 57.7]",Right Foot,Normal,NaN,Open Play,Regular Play,NaN,Blocked,0.011882
1,3939988,1e67056c-794f-4f1a-98b1-12a347a1dfb6,Mexico,César Saúl Huerta Valera,"[98.2, 37.5]",Left Foot,Half Volley,True,Open Play,Regular Play,True,Off T,0.057859
2,3939988,9265e407-3347-4c35-8e52-6b3811cf48f7,Mexico,Orbelín Pineda Alvarado,"[102.0, 57.9]",Right Foot,Normal,NaN,Open Play,Regular Play,NaN,Blocked,0.016745
3,3939988,955126e9-b026-4a35-a255-d4bf81b545f7,Ecuador,Moisés Isaac Caicedo Corozo,"[94.3, 30.3]",Right Foot,Normal,NaN,Free Kick,From Free Kick,NaN,Blocked,0.053783
4,3939988,3b4bdfd2-d26f-4244-bf75-aff5a7a5ff76,Ecuador,Ray Kendry Páez Andrade,"[92.6, 34.4]",Left Foot,Half Volley,True,Open Play,From Free Kick,NaN,Saved,0.017840


<class 'pandas.DataFrame'>
RangeIndex: 12374 entries, 0 to 12373
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   match_id           12374 non-null  int64  
 1   id                 12374 non-null  str    
 2   team               12374 non-null  str    
 3   player             12374 non-null  str    
 4   location           12374 non-null  str    
 5   shot_body_part     12374 non-null  str    
 6   shot_technique     12374 non-null  str    
 7   shot_first_time    3880 non-null   object 
 8   shot_type          12374 non-null  str    
 9   play_pattern       12374 non-null  str    
 10  under_pressure     3096 non-null   object 
 11  shot_outcome       12374 non-null  str    
 12  shot_statsbomb_xg  12374 non-null  float64
dtypes: float64(1), int64(1), object(2), str(9)
memory usage: 1.2+ MB


# 1.1 Handling Missing Values

Missing values were identified in the **shot_first_time** and **under_pressure** attributes. Since these attributes represent Boolean flags, missing values are treated as False, indicating that the corresponding condition was not recorded as present. This approach preserves all shot records without removing potentially useful training data.

In [8]:
# Check missing values in each column
print("Missing values before cleaning:")
print(df.isnull().sum())


Missing values before cleaning:
match_id                0
id                      0
team                    0
player                  0
location                0
shot_body_part          0
shot_technique          0
shot_first_time      8494
shot_type               0
play_pattern            0
under_pressure       9278
shot_outcome            0
shot_statsbomb_xg       0
dtype: int64


In [16]:
# Replace missing values with False
df["shot_first_time"] = df["shot_first_time"].fillna(False)
df["under_pressure"] = df["under_pressure"].fillna(False)


print("Missing values after cleaning:")
print(df.isnull().sum())

# Check the data types
print(df[["shot_first_time", "under_pressure"]].dtypes)


Missing values after cleaning:
match_id             0
id                   0
team                 0
player               0
location             0
shot_body_part       0
shot_technique       0
shot_first_time      0
shot_type            0
play_pattern         0
under_pressure       0
shot_outcome         0
shot_statsbomb_xg    0
dtype: int64
shot_first_time    object
under_pressure     object
dtype: object


# 1.2 Creating the Binary Target Variable

The **shot_outcome** column contains different shot outcomes. Since this project is a binary classification task, a new target variable named goal is created, where 1 represents a goal and 0 represents any non-goal outcome.

In [10]:

# Check the original shot outcomes
print("Original shot outcomes:")
print(df["shot_outcome"].value_counts())

# binary target variable
df["goal"] = (df["shot_outcome"] == "Goal").astype(int)

# the new target variable
print("\nBinary target distribution:")
print(df["goal"].value_counts())

# Display sample records
display(df[["shot_outcome", "goal"]].head())


Original shot outcomes:
shot_outcome
Off T               3932
Blocked             3079
Saved               2940
Goal                1383
Wayward              672
Post                 281
Saved Off Target      53
Saved to Post         34
Name: count, dtype: int64

Binary target distribution:
goal
0    10991
1     1383
Name: count, dtype: int64


,shot_outcome,goal
0,Blocked,0
1,Off T,0
2,Blocked,0
3,Blocked,0
4,Saved,0


# 1.3 Data Quality Verification

After handling missing values and creating the binary target variable, the dataset is checked for data types, duplicate records, and target class distribution. These checks ensure data consistency and help identify potential issues before model training.

In [11]:
# Check data types
print("Data types:")
print(df.dtypes)

# Check duplicate rows and shot IDs
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate shot IDs:", df["id"].duplicated().sum())

# Check class distribution
print("\nClass distribution (counts):")
print(df["goal"].value_counts())

print("\nClass distribution (percentages):")
print((df["goal"].value_counts(normalize=True) * 100).round(2))

# Check final dataset shape
print("\nDataset shape:", df.shape)


Data types:
match_id               int64
id                       str
team                     str
player                   str
location                 str
shot_body_part           str
shot_technique           str
shot_first_time       object
shot_type                str
play_pattern             str
under_pressure        object
shot_outcome             str
shot_statsbomb_xg    float64
goal                   int64
dtype: object

Duplicate rows: 0
Duplicate shot IDs: 0

Class distribution (counts):
goal
0    10991
1     1383
Name: count, dtype: int64

Class distribution (percentages):
goal
0    88.82
1    11.18
Name: proportion, dtype: float64

Dataset shape: (12374, 14)


# 1.4 Reusable Data-Cleaning Function

A reusable function is created to combine the preprocessing steps into a single process. The function handles missing Boolean values and creates the binary target variable while preserving the original dataset. This ensures that the same cleaning steps can be applied consistently whenever needed.

In [12]:
def clean_shot_data(raw_df):
    # Create a copy to preserve the original dataset
    cleaned_df = raw_df.copy()

    # Handle missing values
    cleaned_df["shot_first_time"] = cleaned_df["shot_first_time"].fillna(False)
    cleaned_df["under_pressure"] = cleaned_df["under_pressure"].fillna(False)

    # Create the binary target variable
    cleaned_df["goal"] = (cleaned_df["shot_outcome"] == "Goal").astype(int)

    return cleaned_df


# Apply the cleaning function to the original dataset
df_clean = clean_shot_data(df_raw)

print("Cleaned dataset shape:", df_clean.shape)
print("Total missing values:", df_clean.isnull().sum().sum())
print("Duplicate shot IDs:", df_clean["id"].duplicated().sum())

display(df_clean.head())


Cleaned dataset shape: (12374, 14)
Total missing values: 0
Duplicate shot IDs: 0


,match_id,id,team,player,location,shot_body_part,shot_technique,shot_first_time,shot_type,play_pattern,under_pressure,shot_outcome,shot_statsbomb_xg,goal
0,3939988,20be3941-09d4-4073-98f7-a56806520e3e,Mexico,Orbelín Pineda Alvarado,"[96.5, 57.7]",Right Foot,Normal,False,Open Play,Regular Play,False,Blocked,0.011882,0
1,3939988,1e67056c-794f-4f1a-98b1-12a347a1dfb6,Mexico,César Saúl Huerta Valera,"[98.2, 37.5]",Left Foot,Half Volley,True,Open Play,Regular Play,True,Off T,0.057859,0
2,3939988,9265e407-3347-4c35-8e52-6b3811cf48f7,Mexico,Orbelín Pineda Alvarado,"[102.0, 57.9]",Right Foot,Normal,False,Open Play,Regular Play,False,Blocked,0.016745,0
3,3939988,955126e9-b026-4a35-a255-d4bf81b545f7,Ecuador,Moisés Isaac Caicedo Corozo,"[94.3, 30.3]",Right Foot,Normal,False,Free Kick,From Free Kick,False,Blocked,0.053783,0
4,3939988,3b4bdfd2-d26f-4244-bf75-aff5a7a5ff76,Ecuador,Ray Kendry Páez Andrade,"[92.6, 34.4]",Left Foot,Half Volley,True,Open Play,From Free Kick,False,Saved,0.017840,0


In [13]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

cleaned_path = processed_dir / "cleaned_shots.csv"
df_clean.to_csv(cleaned_path, index=False)

print("Saved cleaned dataset:", cleaned_path)
print("Shape:", df_clean.shape)

Saved cleaned dataset: ../data/processed/cleaned_shots.csv
Shape: (12374, 14)
